In [ ]:
# !uv add --upgrade --no-cache git+https://github.com/rongardF/tvdatafeed.git

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys

sys.path.append("..")

In [ ]:
from tvDatafeed import TvDatafeed, Interval

# Initialize TvDatafeed (you might need to provide username/password if you have a TradingView account and want to fetch more data)
tv = TvDatafeed()

In [ ]:
# Get historical data for BTCUSDT from BINANCE
# Adjust interval and n_bars as needed
xauusd_data = tv.get_hist(symbol='XAUUSD', exchange='PEPPERSTONE', interval=Interval.in_1_hour, n_bars=10000)

if xauusd_data is None:
    raise Exception("Could not retrieve data. Please check symbol, exchange, and your internet connection.")

In [ ]:
# Display the last few rows of the data
print(xauusd_data.tail())

In [ ]:
xauusd_data = xauusd_data.rename(columns={
    "open": "Open",
    "high": "High",
    "low": "Low",
    "close": "Close",
    "volume": "Volume",
})

In [ ]:
import pandas as pd


def SMA(values, n):
    """
    Return simple moving average of `values`, at
    each step taking into account `n` previous values.
    """
    return pd.Series(values).rolling(n).mean()

In [ ]:
from backtesting import Strategy
from backtesting.lib import crossover


class SmaCross(Strategy):
    # Define the two MA lags as *class variables*
    # for later optimization
    n1 = 7
    n2 = 108
    
    def init(self):
        # Precompute the two moving averages
        self.sma1 = self.I(SMA, self.data.Close, self.n1)
        self.sma2 = self.I(SMA, self.data.Close, self.n2)
    
    def next(self):
        # If sma1 crosses above sma2, close any existing
        # short trades, and buy the asset
        if crossover(self.sma1, self.sma2):
            self.position.close()
            self.buy()

        # Else, if sma1 crosses below sma2, close any existing
        # long trades, and sell the asset
        elif crossover(self.sma2, self.sma1):
            self.position.close()
            self.sell()

In [ ]:
from backtesting import Backtest

bt = Backtest(xauusd_data, SmaCross, cash=10_000, commission=.002)
stats = bt.run()
stats

In [ ]:
bt.plot()

## What

In [ ]:
# === Backtesting Setup ===
from backtesting import Strategy, Backtest
from backtesting.lib import crossover
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')


# === EMA/RSI Strategy ===
class EMARSI_Strategy(Strategy):
    """
    EMA(20,50) Crossover + RSI(14) Confirmation
    - Go Long: EMA(20) crosses above EMA(50) AND RSI > 50
    - Go Short: EMA(20) crosses below EMA(50) AND RSI < 50
    - Exit Long: Price crosses below EMA(20) OR RSI < 30
    - Exit Short: Price crosses above EMA(20) OR RSI > 70
    """
    ema_fast = 20
    ema_slow = 50
    rsi_period = 14
    rsi_upper = 70
    rsi_lower = 30
    
    def init(self):
        close = self.data.Close
        self.ema_fast = self.I(
            lambda x: pd.Series(x).ewm(span=self.ema_fast, adjust=False).mean(),
            close,
            name='EMA_fast'
        )
        self.ema_slow = self.I(
            lambda x: pd.Series(x).ewm(span=self.ema_slow, adjust=False).mean(),
            close,
            name='EMA_slow'
        )
        self.rsi = self.I(
            lambda x: self._compute_rsi(x, self.rsi_period),
            close,
            name='RSI'
        )
    
    def _compute_rsi(self, prices, period=14):
        prices = np.array(prices)
        deltas = np.diff(prices, prepend=prices[0])
        gains = np.where(deltas > 0, deltas, 0)
        losses = np.where(deltas < 0, -deltas, 0)
        avg_gain = np.mean(gains[:period])
        avg_loss = np.mean(losses[:period])
        for i in range(period, len(prices)):
            avg_gain = (avg_gain * (period - 1) + gains[i]) / period
            avg_loss = (avg_loss * (period - 1) + losses[i]) / period
        rs = avg_gain / avg_loss if avg_loss != 0 else 0
        return 100 - (100 / (1 + rs))
    
    def next(self):
        ema_f = self.ema_fast
        ema_s = self.ema_slow
        rsi = self.rsi
        price = self.data.Close
        
        # Check for crossover signals
        long_signal = crossover(ema_f, ema_s) and rsi > 50
        short_signal = crossover(ema_s, ema_f) and rsi < 50
        
        # Position sizing: 10% per trade
        size = 0.10
        
        # Exit conditions
        exit_long = crossover(ema_s, ema_f) or rsi < 30
        exit_short = crossover(ema_f, ema_s) or rsi > 70
        
        # Trading logic
        if not self.position:
            if long_signal:
                self.buy(size=size)
            elif short_signal:
                self.sell(size=size)
        else:
            if self.position.is_long and exit_long:
                self.position.close()
            elif self.position.is_short and exit_short:
                self.position.close()

print('Strategy class defined successfully')

In [ ]:
# === Prepare Data ===
# Convert xauusd_data to backtesting.py format
data = xauusd_data.copy()
data = data[['open', 'high', 'low', 'close', 'volume']]
data.columns = ['Open', 'High', 'Low', 'Close', 'Volume']

print(f'Data shape: {data.shape}')
print(f'Date range: {data.index[0]} to {data.index[-1]}')
print(f'Total hours: {len(data)}')

In [ ]:
# === Walk-Forward Analysis Setup ===
# Parameters
TRAIN_DAYS = 180  # 6 months
TEST_DAYS = 30    # 1 month

# Calculate approximate hours per period (24h for crypto, but market hours for forex/gold)
# XAUUSD trades ~23h/day (gaps on weekend)
HOURS_PER_DAY = 23
n_train = TRAIN_DAYS * HOURS_PER_DAY
n_test = TEST_DAYS * HOURS_PER_DAY
step = n_test  # Roll forward by 1 month

# Generate walk-forward splits
splits = []
start_idx = 0
while start_idx + n_train + n_test <= len(data):
    end_idx = start_idx + n_train + n_test
    train_data = data.iloc[start_idx : start_idx + n_train]
    test_data = data.iloc[start_idx + n_train : end_idx]
    splits.append((train_data, test_data))
    start_idx += step

print(f'Total splits: {len(splits)}')
print(f'Training: ~{TRAIN_DAYS} days ({n_train} bars)')
print(f'Testing: ~{TEST_DAYS} days ({n_test} bars)')

In [ ]:
# === Run Walk-Forward Backtests ===
all_results = []
all_equity_curves = []

for i, (train_data, test_data) in enumerate(splits):
    print(f"\n=== Split {i+1}/{len(splits)} ===")
    print(f"Train: {train_data.index[0].date()} to {train_data.index[-1].date()} ({len(train_data)} bars)")
    print(f"Test:  {test_data.index[0].date()} to {test_data.index[-1].date()} ({len(test_data)} bars)")
    
    # Create backtester
    bt = Backtest(
        test_data,
        EMARSI_Strategy,
        cash=100_000,
        commission=0.001,
        exclusive_orders=True
    )
    
    # Run
    stats = bt.run()
    all_results.append(stats)
    all_equity_curves.append(stats._equity_curve)
    
    print(f"  Sharpe: {stats['Sharpe Ratio']:.2f}")
    print(f"  Return: {stats['Return [%]']:.2f}%")
    print(f"  Max DD: {stats['Max. Drawdown [%]']:.2f}%")
    print(f"  Trades: {len(stats._trades)}")

print("\n=== Walk-Forward Complete ===")

In [ ]:
# === Aggregate Results ===
import matplotlib.pyplot as plt

# Extract metrics
sharpes = [s['Sharpe Ratio'] for s in all_results]
returns = [s['Return [%]'] for s in all_results]
max_dds = [s['Max. Drawdown [%]'] for s in all_results]
sortinos = [s['Sortino Ratio'] for s in all_results]
win_rates = [s['Win Rate [%]'] for s in all_results]
num_trades = [len(s._trades) for s in all_results]

print('=' * 70)
print('WALK-FORWARD RESULTS SUMMARY')
print('=' * 70)
print(f"{'Split':<6} {'Test Period':<25} {'Sharpe':>8} {'Return%':>10} {'MaxDD%':>10} {'Trades':>7}")
print('-' * 70)

for i, (r, s) in enumerate(zip(all_results, splits)):
    test_period = f"{s[1].index[0].date()} to {s[1].index[-1].date()}"
    print(f"{i+1:<6} {test_period:<25} {sharpes[i]:>8.2f} {returns[i]:>10.2f} {max_dds[i]:>10.2f} {num_trades[i]:>7}")

print('-' * 70)
print(f"{'MEAN':<6} {'':<25} {np.mean(sharpes):>8.2f} {np.mean(returns):>10.2f} {np.mean(max_dds):>10.2f} {np.mean(num_trades):>7.1f}")
print(f"{'STD':<6} {'':<25} {np.std(sharpes):>8.2f} {np.std(returns):>8.2f} {np.std(max_dds):>8.2f} {np.std(num_trades):>7.1f}")
print('=' * 70)

In [ ]:
# === Monte Carlo Simulation ===
# Bootstrap from actual walk-forward returns
n_sims = 5000
simulated_returns = []

for _ in range(n_sims):
    # Sample with replacement from actual returns
    sampled_return = np.random.choice(returns)
    simulated_returns.append(sampled_return)

simulated_returns = np.array(simulated_returns)

# Calculate confidence intervals
ci_lower = np.percentile(simulated_returns, 5)
ci_upper = np.percentile(simulated_returns, 95)
prob_loss = (simulated_returns < 0).mean() * 100

print('=' * 50)
print('MONTE CARLO ANALYSIS (5000 simulations)')
print('=' * 50)
print(f"Expected Return: {np.mean(simulated_returns):.2f}%")
print(f"95% CI Lower:    {ci_lower:.2f}%")
print(f"95% CI Upper:    {ci_upper:.2f}%")
print(f"Probability of Loss: {prob_loss:.1f}%")
print(f"Worst Case (5th %): {np.percentile(simulated_returns, 5):.2f}%")
print('=' * 50)

In [ ]:
# === Visualization ===
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Equity curves
ax1 = axes[0, 0]
for i, eq in enumerate(all_equity_curves):
    ax1.plot(eq.index, eq.values / eq.values[0] * 100_000, alpha=0.7, label=f'Split {i+1}')
ax1.set_title('Equity Curves (Normalized to $100k)')
ax1.set_xlabel('Date')
ax1.set_ylabel('Equity ($)')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# 2. Sharpe ratio bar chart
ax2 = axes[0, 1]
colors = ['green' if s > 0 else 'red' for s in sharpes]
ax2.bar(range(1, len(sharpes)+1), sharpes, color=colors, alpha=0.7)
ax2.axhline(y=np.mean(sharpes), color='black', linestyle='--', label=f'Mean: {np.mean(sharpes):.2f}')
ax2.set_title('Sharpe Ratio by Split')
ax2.set_xlabel('Split')
ax2.set_ylabel('Sharpe Ratio')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Returns vs Max Drawdown scatter
ax3 = axes[1, 0]
scatter = ax3.scatter(max_dds, returns, s=100, c=range(len(returns)), cmap='viridis')
for i, (dd, ret) in enumerate(zip(max_dds, returns)):
    ax3.annotate(f'S{i+1}', (dd, ret), fontsize=9, ha='center')
ax3.axhline(y=0, color='red', linestyle='-', alpha=0.5)
ax3.set_title('Return vs Max Drawdown')
ax3.set_xlabel('Max Drawdown (%)')
ax3.set_ylabel('Return (%)')
ax3.grid(True, alpha=0.3)

# 4. Monte Carlo histogram
ax4 = axes[1, 1]
ax4.hist(simulated_returns, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
ax4.axvline(x=np.mean(simulated_returns), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(simulated_returns):.2f}%')
ax4.axvline(x=ci_lower, color='orange', linestyle='--', linewidth=2, label=f'5th %: {ci_lower:.2f}%')
ax4.axvline(x=ci_upper, color='green', linestyle='--', linewidth=2, label=f'95th %: {ci_upper:.2f}%')
ax4.axvline(x=0, color='black', linestyle='-', linewidth=1)
ax4.set_title('Monte Carlo Return Distribution')
ax4.set_xlabel('Return (%)')
ax4.set_ylabel('Frequency')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# === Final Summary ===
print('=' * 60)
print('FINAL BACKTEST SUMMARY')
print('=' * 60)
print(f'Strategy: EMA(20,50) Crossover + RSI(14) Confirmation')
print(f'Position Sizing: {POSITION_SIZE*100}% per trade')
print(f'Walk-Forward: {TRAIN_DAYS} days train / {TEST_DAYS} days test')
print(f'Number of Splits: {len(splits)}')
print('-' * 60)
print(f"Mean Sharpe Ratio:    {np.mean(sharpes):.2f}")
print(f"Mean Return:          {np.mean(returns):.2f}%")
print(f"Mean Max Drawdown:    {np.mean(max_dds):.2f}%")
print(f"Mean Win Rate:        {np.mean(win_rates):.1f}%")
print(f"Mean Sortino Ratio:   {np.mean(sortinos):.2f}")
print('-' * 60)
print(f"Monte Carlo Expected Return: {np.mean(simulated_returns):.2f}%")
print(f"Monte Carlo 95% CI: ({ci_lower:.2f}%, {ci_upper:.2f}%)")
print(f"Probability of Loss: {prob_loss:.1f}%")
print('=' * 60)